In [ ]:
# Import all necessary libraries
import gzip
import os
import csv
import math
import boto3
import random
import pandas as pd
import polars as pl
from datetime import datetime, timedelta

# Extract Positive subjects with re-admission within 10 days apart

In [ ]:
# Extract patient and admission data

demographic = pd.read_csv('/Users/ketki/Downloads/mimic-iii-clinical-database-1.4/PATIENTS.csv.gz', compression='gzip')
# demographic.head()
df = pd.read_csv('/Users/ketki/Downloads/mimic-iii-clinical-database-1.4/ADMISSIONS.csv.gz', compression='gzip')
# df.head()

In [ ]:
# 10-Day Readmission Sample Reconstruction
# Positive Samples  : Patients readmitted within 10 days
# Negative Samples  : Patients without 10-day readmission

# Convert admission and discharge times to datetime
df['ADMITTIME'] = pd.to_datetime(df['ADMITTIME'])
df['DISCHTIME'] = pd.to_datetime(df['DISCHTIME'])

# Sort admissions chronologically per patient
admissions = df.sort_values(by=['SUBJECT_ID', 'ADMITTIME'])

# Get previous discharge time and previous HADM_ID per patient
admissions['prev_DISCHTIME'] = admissions.groupby('SUBJECT_ID')['DISCHTIME'].shift(1)
admissions['prev_HADM_ID'] = admissions.groupby('SUBJECT_ID')['HADM_ID'].shift(1)

# Calculate days between consecutive admissions
admissions['days_since_last_discharge'] = (
    admissions['ADMITTIME'] - admissions['prev_DISCHTIME']
).dt.days

# Positive Triplet Extraction (≤ 10-day readmission)
readmit_within_10 = admissions[
    (admissions['days_since_last_discharge'] > 0) & 
    (admissions['days_since_last_discharge'] <= 10)
]

positive_triplets = readmit_within_10[
    ['SUBJECT_ID', 'prev_HADM_ID', 'HADM_ID']
].rename(columns={
    'prev_HADM_ID': 'HADM_ID_1',
    'HADM_ID': 'HADM_ID_2'
})

positive_triplet_list = [tuple(x) for x in positive_triplets.to_numpy()]

print(f"Number of positive 10-day readmission triplets: {len(positive_triplet_list)}")
print(positive_triplet_list[:10])


# Negative Triplet Extraction (No 10-day readmission)
positive_subjects = set(positive_triplets['SUBJECT_ID'])
all_subjects = set(admissions['SUBJECT_ID'])
negative_subjects = list(all_subjects - positive_subjects)

random_neg_subjects = random.sample(negative_subjects, 20)

neg_df = admissions[admissions['SUBJECT_ID'].isin(random_neg_subjects)]

neg_pairs = neg_df.groupby('SUBJECT_ID').last().reset_index()
neg_pairs = neg_pairs[['SUBJECT_ID', 'HADM_ID']]

neg_pairs = neg_pairs.rename(columns={'HADM_ID': 'HADM_ID_1'})
neg_pairs['HADM_ID_2'] = None  # Explicitly indicate no readmission

negative_triplet_list = [tuple(x) for x in neg_pairs.to_numpy()]

print(f"Number of negative triplets (no 10-day readmission): {len(negative_triplet_list)}")
print(negative_triplet_list[:10])

Number of triplets: 1729
[(36.0, 182104.0, 122659.0), (68.0, 170467.0, 108329.0), (103.0, 130744.0, 133550.0), (109.0, 170149.0, 147469.0), (109.0, 131345.0, 139061.0), (109.0, 139061.0, 172335.0), (109.0, 161950.0, 173633.0), (109.0, 173633.0, 140167.0), (109.0, 113189.0, 158995.0), (109.0, 151240.0, 102024.0)]


In [ ]:
# Extract only the last triplet per subject 
# triplets = positive_triplet_list / negative_triplet_list

latest_triplets = triplets.sort_values(
    by=['SUBJECT_ID', 'HADM_ID_2']
).groupby('SUBJECT_ID').tail(1)

print("Number of latest triplets per subject:", len(latest_triplets))

# Define pair of subject and their first admission for downstream analysis
pairs = latest_triplets[['SUBJECT_ID', 'HADM_ID_1']].copy()

Number of latest triplets per subject: 1523


# Extract the Clinical notes for positive and negative samples

In [ ]:
directory = '.'

data = pd.read_csv('/Users/ketki/Downloads/mimic-iii-clinical-database-1.4/NOTEEVENTS.csv.gz', compression='gzip')
data['text_length'] = data['TEXT'].apply(lambda x:len(str(x)))
# data

/var/folders/nt/t2rcl5mn1kl1k_dtq25_f1900000gn/T/ipykernel_20027/3907201887.py:3: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/Users/ketki/Downloads/mimic-iii-clinical-database-1.4/NOTEEVENTS.csv.gz', compression='gzip')


,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,text_length
0,174,22532,167853.0,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...,889
1,175,13702,107527.0,2118-06-14,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-6-2**] Discharg...,12633
2,176,13702,167118.0,2119-05-25,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2119-5-4**] D...,9736
3,177,13702,196489.0,2124-08-18,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2124-7-21**] ...,17138
4,178,26880,135453.0,2162-03-25,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2162-3-3**] D...,14499
...,...,...,...,...,...,...,...,...,...,...,...,...
2083175,2070657,31097,115637.0,2132-01-21,2132-01-21 03:27:00,2132-01-21 03:38:00,Nursing/other,Report,17581.0,NaN,NPN\n\n\n#1 Infant remains in RA with O2 sats...,878
2083176,2070658,31097,115637.0,2132-01-21,2132-01-21 09:50:00,2132-01-21 09:53:00,Nursing/other,Report,19211.0,NaN,"Neonatology\nDOL #5, CGA 36 weeks.\n\nCVR: Con...",634
2083177,2070659,31097,115637.0,2132-01-21,2132-01-21 16:42:00,2132-01-21 16:44:00,Nursing/other,Report,20104.0,NaN,Family Meeting Note\nFamily meeting held with ...,563
2083178,2070660,31097,115637.0,2132-01-21,2132-01-21 18:05:00,2132-01-21 18:16:00,Nursing/other,Report,16023.0,NaN,NPN 1800\n\n\n#1 Resp: [**Known lastname 2243*...,1995


In [ ]:
# Fetch different categories of notes recorded
unique_categories = data['CATEGORY'].unique()
print("Unique CATEGORY values:")
print(unique_categories)

print("\nNumber of unique CATEGORY values:", len(unique_categories))

unique_descriptions = data['DESCRIPTION'].unique()
print("\nUnique DESCRIPTION values:")
print(unique_descriptions)

print("\nNumber of unique DESCRIPTION values:", len(unique_descriptions))

Unique CATEGORY values:
['Discharge summary' 'Echo' 'ECG' 'Nursing' 'Physician ' 'Rehab Services'
 'Case Management ' 'Respiratory ' 'Nutrition' 'General' 'Social Work'
 'Pharmacy' 'Consult' 'Radiology' 'Nursing/other']

Number of unique CATEGORY values: 15

Unique DESCRIPTION values:
['Report' 'Addendum' 'Nursing Transfer Note' ...
 'PLACE CATH CAROTID/INOM ART' 'L US MSK ASPIRATE/INJ GANGLION CYST LEFT'
 'RO HIP NAILING IN OR W/FILMS & FLUORO RIGHT IN O.R.']

Number of unique DESCRIPTION values: 3848


In [ ]:
# This is currently not used but a part fo analysis
# Extract average number of longitudinal notes per subject admission pair and filter triplets above set threshold
note_counts = (
    data.groupby(['SUBJECT_ID', 'HADM_ID'])
        .size()
        .reset_index(name='note_count')
)

pairs_with_notes = pairs.merge(
    note_counts,
    how='left',
    left_on=['SUBJECT_ID', 'HADM_ID_1'],
    right_on=['SUBJECT_ID', 'HADM_ID']
)

pairs_with_notes['note_count'] = pairs_with_notes['note_count'].fillna(0)

average_notes = pairs_with_notes['note_count'].mean()
print("Average number of notes for Admission 1 across latest triplets:", average_notes)

threshold = math.floor(average_notes)
print("Rounded-down note threshold:", threshold)

filtered_pairs = pairs_with_notes[pairs_with_notes['note_count'] > threshold]

filtered_triplets = latest_triplets.merge(
    filtered_pairs[['SUBJECT_ID', 'HADM_ID_1']],
    on=['SUBJECT_ID', 'HADM_ID_1'],
    how='inner'
)

print("Number of triplets after filtering:", len(filtered_triplets))
filtered_triplets.head()

In [ ]:
# Filter only notes with CATEGORY as Discharge summary
discharge_notes = data[data['CATEGORY'] == 'Discharge summary']

# Get all pairs where tha admission has at least one discharge summary
hads_with_discharge = discharge_notes['HADM_ID'].unique()

# Filter triplets to keep only those where both admissions have at least one discharge summary
filtered_triplets_ds = filtered_triplets[
    (filtered_triplets['HADM_ID_1'].isin(hads_with_discharge)) &
    (filtered_triplets['HADM_ID_2'].isin(hads_with_discharge))
].copy()

print("Number of triplets after filtering for discharge summaries:", len(filtered_triplets_ds))

Number of triplets after filtering for discharge summaries: 363


In [ ]:
# Filter those triplets where there is clinical relevance between the two admissions using LLM Validation 

os.makedirs("triplet", exist_ok=True)
# Set the API key as an environment variable
os.environ['AWS_BEARER_TOKEN_BEDROCK'] = "<your_bedrock_token>"

# Create the Bedrock client
client = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1"
)

# Define the model and message
model_id = "us.anthropic.claude-3-5-haiku-20241022-v1:0"
messages = [{"role": "user", "content": [{"text": prompt}]}]

# Randomly select triplets
sampled_triplets = filtered_triplets.sample(n=15)

# Loop through each triplet
for idx, row in sampled_triplets.iterrows():
    subject = row["SUBJECT_ID"]
    hadm1 = row["HADM_ID_1"]
    hadm2 = row["HADM_ID_2"]
    print('subject',subject)
    print('hadm1',hadm1)
    print('hadm2',hadm2)
    # Fetch Discharge summary notes for Admission 1
    notes1_df = data[
        (data["SUBJECT_ID"] == subject) &
        (data["HADM_ID"] == hadm1) &
        (data["CATEGORY"] == "Discharge summary")
    ][["CATEGORY", "DESCRIPTION", "TEXT"]].dropna(subset=["TEXT"])
    
    notes1 = [
        f"CATEGORY: {cat}\nDESCRIPTION: {desc}\n\n{txt}"
        for cat, desc, txt in zip(notes1_df["CATEGORY"], notes1_df["DESCRIPTION"], notes1_df["TEXT"])
    ]

    # Fetch Discharge summary notes for Admission 2
    notes2_df = data[
        (data["SUBJECT_ID"] == subject) &
        (data["HADM_ID"] == hadm2) &
        (data["CATEGORY"] == "Discharge summary")
    ][["CATEGORY", "DESCRIPTION", "TEXT"]].dropna(subset=["TEXT"])
    
    notes2 = [
        f"CATEGORY: {cat}\nDESCRIPTION: {desc}\n\n{txt}"
        for cat, desc, txt in zip(notes2_df["CATEGORY"], notes2_df["DESCRIPTION"], notes2_df["TEXT"])
    ]

    # Design a prompt to check for clinical relevance between the two discharge summaries
    combined_text1 = notes1
    combined_text2 = notes2
    print('Discharge summary admission 1:', combined_text1)
    print('Discharge summary admission 1:', combined_text2)
    prompt = f"1st discharge note: \n {combined_text1} \n\n\n\n readmission discharge note: \n {combined_text2} \n\n\n\n You are given 1st admission and readmission discharge notes \n 1. ⁠Check if there is a instruction in the fisrt note that asks the patient to get readmitted again and return False.\n 2. ⁠If the previous step result is True, check if there is any ailment or symptom in the 1st note that were left untreated that could have caused the readmission. Give a binary True or False answer if there was something that could have been done to prevent the readmission."

    # Make the API call
    response = client.converse(
        modelId=model_id,
        messages=messages)

    # Print the response
    print(response['output']['message']['content'][0]['text'])


subject 15716.0
hadm1 114110.0
hadm2 107629.0
Discharge summary admission 1: ["CATEGORY: Discharge summary\nDESCRIPTION: Report\n\nAdmission Date:  [**2147-7-23**]              Discharge Date:   [**2147-8-11**]\n\nDate of Birth:  [**2077-11-4**]             Sex:   F\n\nService: MEDICINE\n\nAllergies:\nPlavix / Heparin Agents\n\nAttending:[**First Name3 (LF) 2485**]\nChief Complaint:\nCC: Hypotension, fever, tachypnea.\n\n\nMajor Surgical or Invasive Procedure:\nParacentesis, G-J tube placement with flouroscopy.\n\nHistory of Present Illness:\nHPI: 69 y/o female with a h/o pansensitive TB on 3 drug regimen,\nMDS with pancytopenia, respiratory failure s/p trach placement\nand revision [**5-/2147**], recently discharged [**2147-6-30**] with sepsis\npresumed [**12-23**] PNA (by MDR E coli, pseudomonas) who comes in from\n[**Hospital3 **] with tachycardia, tachypnea, and hypotension.\nTwo days ago the patient's G-tube fell out and she had it\nreplaced yesterday. She also had a new PICC tube

In [ ]:
# Create a file containing all the Discharge Summary data
file_path = "final_data.csv"

# Create file with headers if it does not exist
if not os.path.exists(file_path):
    with open(file_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            'SUB_ID',
            'ADM_ID_1',
            'ADM_ID_2',
            'DISCHARGE_NOTE_1',
            'DISCHARGE_NOTE_2',
            'LLM_OUTPUT'
        ])

# Append row safely
with open(file_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL) 
    writer.writerow([
        subject,
        hadm1,
        hadm2,
        combined_text1,
        combined_text2,
        response["output"]["message"]["content"][0]["text"]
    ])

# Extract Demographic of for positive and negative samples

In [ ]:
# Open csv containign stored positive triplet with their discharge summaries
df = pd.read_csv('final_data.csv')
demo = pd.read_csv('/Users/ketki/Downloads/mimic-iii-clinical-database-1.4/PATIENTS.csv.gz', compression='gzip')
# Match SUBJECT_ID in df with demo to get DOB and SEX columns from demo and add to df
df = df.merge(demo[['SUBJECT_ID', 'DOB', 'GENDER']], left_on='SUB_ID', right_on='SUBJECT_ID', how='left')
df = df.drop(columns=['SUBJECT_ID'])
# From ADMISSIONS get ADMITTIME for ADM_ID_1 and add it to df as ADM_TIME_1
admissions = pd.read_csv('/Users/ketki/Downloads/mimic-iii-clinical-database-1.4/ADMISSIONS.csv.gz', compression='gzip')
admissions_subset = admissions[['HADM_ID', 'ADMITTIME']]
admissions_subset = admissions_subset.rename(columns={'HADM_ID': 'ADM_ID_1', 'ADMITTIME': 'ADM_TIME_1'})
df = df.merge(admissions_subset, on='ADM_ID_1', how='left')

#For each row in df, calculate age at ADM_TIME_1 using DOB and ADM_TIME_1 and add it to df as AGE_AT_ADM_1
df['DOB'] = pd.to_datetime(df['DOB'])
df['ADM_TIME_1'] = pd.to_datetime(df['ADM_TIME_1'])
df['AGE_AT_ADM_1'] = (df['ADM_TIME_1'] - df['DOB']).dt.days / 365.25
df['AGE_AT_ADM_1'] = df['AGE_AT_ADM_1'].astype(int)
df = df.drop(columns=['DOB', 'ADM_TIME_1'])
df.head()
df.to_csv("final_data.csv", index=False)

,SUB_ID,ADM_ID_1,ADM_ID_2,DISCHARGE_NOTE_1,DISCHARGE_NOTE_2,LLM_OUTPUT,GENDER,AGE_AT_ADM_1
0,17384,118828,190248,"[""CATEGORY: Discharge summary\nDESCRIPTION: Re...","[""CATEGORY: Discharge summary\nDESCRIPTION: Re...",Let me analyze the discharge notes systematica...,F,36
1,76418,152251,114836,"[""CATEGORY: Discharge summary\nDESCRIPTION: Re...","[""CATEGORY: Discharge summary\nDESCRIPTION: Re...",Let me analyze this systematically:\n\n1. Firs...,M,62
2,6131,147548,156448,"[""CATEGORY: Discharge summary\nDESCRIPTION: Re...","[""CATEGORY: Discharge summary\nDESCRIPTION: Re...",Let me analyze the discharge notes step by ste...,M,68
3,21990,102865,171034,['CATEGORY: Discharge summary\nDESCRIPTION: Re...,['CATEGORY: Discharge summary\nDESCRIPTION: Re...,"Let's analyze this step by step:\n\n1. First, ...",F,75
4,6131,147548,156448,"[""CATEGORY: Discharge summary\nDESCRIPTION: Re...","[""CATEGORY: Discharge summary\nDESCRIPTION: Re...",Let's analyze this systematically:\n\n1. First...,M,68


# Extracting Patient Vitals Timeseries for positive and negative samples

In [ ]:
allowed_values = [
    618, 51, 211, 442, 455, 645, 646, 676, 678, 3288,
    3313, 3315, 3321, 3323, 3337, 8113, 5896, 3837, 3838,
    2933, 3603, 3652, 3654, 3655, 7260, 7293, 8364, 8368,
    8440, 8441, 8498, 8502, 8503, 8506, 8555, 6643, 6701,
    228151, 228152, 224027, 224167, 227054, 227242, 227243,
    224767, 224769, 227916, 227918, 227919, 224643, 220045,
    220050, 220051, 220179, 220180, 220210, 220227, 227858,
    227860, 227861, 223851, 220277, 223761, 223762, 225309,
    225310, 220181
]
d_items = pd.read_csv("D_ITEMS.csv")

patients = pd.read_csv("final_data.csv") 

for i, row in patients.iterrows():

    sub_id = row['SUB_ID']
    adm_id = row['ADM_ID_1']

    print(f"Processing SUB={sub_id}, ADM={adm_id}")

    # read in chunks & filter
    result_chunks = []
    for chunk in pd.read_csv(
        "/Users/ketki/Downloads/mimic-iii-clinical-database-1.4/CHARTEVENTS.csv.gz",
        compression='gzip',
        chunksize=100_000
    ):
        result_chunks.append(
            chunk[(chunk["SUBJECT_ID"] == sub_id) & (chunk["HADM_ID"] == adm_id)]
        )

    result = pd.concat(result_chunks, ignore_index=True)

    # filter desired items
    df_filtered = result[result['ITEMID'].isin(allowed_values)]
    df_filtered = df_filtered.merge(d_items[['ITEMID', 'LABEL']], on='ITEMID')

    # drop unused columns
    drop_cols = ['ICUSTAY_ID','ROW_ID','STORETIME','CGID','VALUENUM',
                 'WARNING','ERROR','RESULTSTATUS','STOPPED']
    df_filtered.drop(columns=drop_cols, inplace=True)

    # group
    df_grouped = (df_filtered
        .pivot_table(index=['SUBJECT_ID', 'CHARTTIME'],
                     columns='LABEL', values='VALUE',
                     aggfunc='first')
        .reset_index())
    
    df_grouped['CHARTTIME'] = pd.to_datetime(df_grouped['CHARTTIME']).dt.floor('4h')
    df_grouped = df_grouped.loc[:, ~df_grouped.columns.duplicated()]
    df_grouped = df_grouped.apply(pd.to_numeric, errors='ignore')
    numeric_cols = df_grouped.select_dtypes(include='number').columns
    df_grouped = df_grouped.loc[:, ~df_grouped.columns.duplicated()]
    df_grouped = df_grouped.groupby(
        ['SUBJECT_ID', 'CHARTTIME'], as_index=False
    )[numeric_cols].mean()

    filename = f"{sub_id}_{adm_id}.csv"
    df_grouped.to_csv(filename, index=False)

    print("Saved:", filename)

Processing SUB=3488, ADM=140358


/var/folders/nt/t2rcl5mn1kl1k_dtq25_f1900000gn/T/ipykernel_8183/2588527495.py:27: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/var/folders/nt/t2rcl5mn1kl1k_dtq25_f1900000gn/T/ipykernel_8183/2588527495.py:64: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_grouped = df_grouped.apply(pd.to_numeric, errors='ignore')


Saved: 3488_140358.csv
Processing SUB=8512, ADM=162973


/var/folders/nt/t2rcl5mn1kl1k_dtq25_f1900000gn/T/ipykernel_8183/2588527495.py:27: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/var/folders/nt/t2rcl5mn1kl1k_dtq25_f1900000gn/T/ipykernel_8183/2588527495.py:64: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_grouped = df_grouped.apply(pd.to_numeric, errors='ignore')


Saved: 8512_162973.csv
Processing SUB=8778, ADM=128848


...

/var/folders/nt/t2rcl5mn1kl1k_dtq25_f1900000gn/T/ipykernel_8183/2588527495.py:27: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/var/folders/nt/t2rcl5mn1kl1k_dtq25_f1900000gn/T/ipykernel_8183/2588527495.py:64: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_grouped = df_grouped.apply(pd.to_numeric, errors='ignore')


Saved: 83449_142566.csv
Processing SUB=90303, ADM=141008


/var/folders/nt/t2rcl5mn1kl1k_dtq25_f1900000gn/T/ipykernel_8183/2588527495.py:27: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


Saved: 90303_141008.csv


/var/folders/nt/t2rcl5mn1kl1k_dtq25_f1900000gn/T/ipykernel_8183/2588527495.py:64: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_grouped = df_grouped.apply(pd.to_numeric, errors='ignore')
